# NLP Preprocessing on Book Descriptions

In [ ]:
import pandas as pd
import nltk
import re
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger_eng')

## Load Data

In [ ]:
books = pd.read_csv("books_cleaned.csv")
books = books.dropna(subset=["description"])
print(f"Total books: {len(books)}")
books[["title", "description"]].head()

## 1. Tokenization

In [ ]:
sample_text = books["description"].iloc[0]
print("Original Text:\n", sample_text[:500])
print("\n--- Word Tokenization ---")
word_tokens = word_tokenize(sample_text)
print(f"Total tokens: {len(word_tokens)}")
print("First 20 tokens:", word_tokens[:20])

print("\n--- Sentence Tokenization ---")
sent_tokens = sent_tokenize(sample_text)
print(f"Total sentences: {len(sent_tokens)}")
for i, sent in enumerate(sent_tokens[:3]):
    print(f"Sentence {i+1}: {sent}")

## 2. Filtration & Stop Word Removal

In [ ]:
stop_words = set(stopwords.words('english'))
print(f"Total English stop words: {len(stop_words)}")
print("Sample stop words:", list(stop_words)[:10])

filtered_tokens = [word for word in word_tokens if word.lower() not in stop_words and word.isalpha()]
print(f"\nBefore filtering: {len(word_tokens)} tokens")
print(f"After filtering: {len(filtered_tokens)} tokens")
print(f"Removed: {len(word_tokens) - len(filtered_tokens)} tokens")
print("\nFirst 20 filtered tokens:", filtered_tokens[:20])

## 3. Stemming

In [ ]:
stemmer = PorterStemmer()

sample_words = ["running", "happiness", "played", "flying", "stories", "beautiful", "imagination"]
print("Stemming Examples:")
for word in sample_words:
    print(f"  {word} \u2192 {stemmer.stem(word)}")

stemmed_tokens = [stemmer.stem(word) for word in filtered_tokens]
print(f"\nFirst 20 stemmed tokens: {stemmed_tokens[:20]}")

## 4. Lemmatization

In [ ]:
lemmatizer = WordNetLemmatizer()

sample_words = ["running", "better", "flies", "stories", "women", "played"]
print("Lemmatization Examples:")
for word in sample_words:
    print(f"  {word} \u2192 {lemmatizer.lemmatize(word, pos='v')}")

lemmatized_tokens = [lemmatizer.lemmatize(word.lower()) for word in filtered_tokens]
print(f"\nFirst 20 lemmatized tokens: {lemmatized_tokens[:20]}")

## 5. Complete Preprocessing Pipeline

In [ ]:
def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    tokens = word_tokenize(text)
    tokens = [word for word in tokens if word not in stop_words and len(word) > 2]
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    return " ".join(tokens)

# Apply to all books
books["processed_description"] = books["description"].apply(preprocess_text)

print("Sample comparison:")
print(f"\nOriginal: {books['description'].iloc[0][:200]}...")
print(f"\nProcessed: {books['processed_description'].iloc[0][:200]}...")

## Save Preprocessed Data

In [ ]:
books.to_csv("books_preprocessed.csv", index=False)
print(f"Saved {len(books)} preprocessed books to books_preprocessed.csv")